# Introduction to LLM APIs: OpenAI and Ollama

## Overview
In this notebook, we will explore how to interact with Large Language Models (LLMs) using Python.
We will cover two main approaches:

1.  **Proprietary Models (OpenAI)**: Using the OpenAI API to access powerful models like GPT-4o.
2.  **Open Source Models (Ollama)**: Running local LLMs (like Llama 3 or Phi-4) on your own machine (or in this case, the Google Colab environment).

By the end of this session, you will understand how to send prompts to these models and receive responses programmatically.


## 1. OpenAI Python API

First, we need to install the official OpenAI Python SDK. This library simplifies making requests to OpenAI's servers.


In [ ]:
!pip install -Uq openai


### API Key Setup

To use OpenAI, you need an API key. In Google Colab, it is best practice to store your keys in the `Secrets` manager (the key icon on the left sidebar).

1.  Click the **key icon** on the left.
2.  Add a new secret named `OPENAI_API_KEY` with your actual key value.
3.  Toggle 'Notebook access' to on.

The code below retrieves this key securely.


In [ ]:
import openai
from google.colab import userdata

try:
    api_key = userdata.get('OPENAI_API_KEY')
except Exception as e:
    print("Error retrieving API key. Make sure you set 'OPENAI_API_KEY' in Colab Secrets.")
    api_key = None


### Basic Completion

Let's make our first call to the API. We will use the `chat.completions.create` method.
This method requires:
-   `model`: The specific model ID (e.g., `gpt-4o-mini`).
-   `messages`: A list of message objects, where each object has a `role` (system, user, assistant) and `content`.


In [ ]:
from openai import OpenAI

if api_key:
    client = OpenAI(api_key=api_key)

    completion = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": "Explain what a Large Language Model is in one sentence."}
        ]
    )
    print(completion.choices[0].message.content)
else:
    print("Skipping OpenAI call because API Key is missing.")


### Streaming Responses

LLMs generate text token by token. Instead of waiting for the full response, we can 'stream' the output so it appears as it is being written. This creates a better user experience.


In [ ]:
if api_key:
    stream = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "user", "content": "Explain how a neural network works to a 5-year-old."}
        ],
        stream=True,  # Enable streaming
    )

    print("Streaming response:")
    for chunk in stream:
        content = chunk.choices[0].delta.content
        if content:
            print(content, end="", flush=True)
else:
    print("Skipping OpenAI call.")


## 2. Ollama (Local LLMs)

[Ollama](https://ollama.com/) is a tool that allows you to run open-source LLMs locally. It simplifies the process of downloading and managing models like Llama 3, Mistral, and Gemma.

### Setting up Ollama in Colab
Since Google Colab is a virtual environment, we need to:
1.  Install Ollama.
2.  Start the Ollama server in the background.
3.  Pull (download) the model we want to use.


In [ ]:
# 1. Install Ollama
!curl -fsSL https://ollama.com/install.sh | sh


In [ ]:
# 2. Start Ollama server in the background using nohup
!nohup ollama serve > ollama.log 2>&1 &

# Wait a few seconds for the server to spin up
import time
time.sleep(5)
print("Ollama server started.")


In [ ]:
# 3. Pull a lightweight model (Llama 3.2 is great for Colab)
!ollama pull llama3.2


### Using Ollama Python Library

Just like OpenAI, Ollama has a Python library to interact with the models running on the local server.


In [ ]:
# Install the Ollama python client
!pip install -q ollama


In [ ]:
import ollama

response = ollama.chat(model='llama3.2', messages=[
  {
    'role': 'user',
    'content': 'Why is the sky blue?'
  },
])
print(response['message']['content'])


## 3. OpenAI Compatibility

One of the coolest features of Ollama is that it is **OpenAI-compatible**.
This means you can use the `openai` python client to talk to your local Ollama models! You just need to change the `base_url` to point to your local server.

**Why is this useful?**
It allows you to switch between expensive proprietary models (OpenAI) and free local models (Ollama) without rewriting your entire application logic.


In [ ]:
from openai import OpenAI

# Point the OpenAI client to the local Ollama server
client = OpenAI(
    base_url='http://localhost:11434/v1',
    api_key='ollama', # Required, but ignored by Ollama
)

response = client.chat.completions.create(
    model="llama3.2",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Who wrote Hamlet?"}
    ]
)
print(response.choices[0].message.content)


# __Student Assignment__

## **Lab: Building an Email Assistant**

**Objective**: Create a python function that helps users draft professional emails using an LLM.

**Instructions**:
1.  Complete the `generate_email` function in the template below.
2.  Use the `openai` client configured for Ollama (or OpenAI if you prefer).
3.  Test your function with different inputs.

**Challenge**:
Add a `tone` parameter to the function (e.g., formal, friendly, urgent) and modify the system prompt to reflect this tone.


In [ ]:
from openai import OpenAI

# Setup client (using Ollama local server for free testing)
client = OpenAI(
    base_url='http://localhost:11434/v1',
    api_key='ollama'
)

def generate_email(subject, recipient_name, additional_info, tone="professional"):
    # TODO: Construct your system and user prompt here
    # Hint: Include the 'tone' in your system instruction
    
    response = client.chat.completions.create(
        model="llama3.2",
        # TODO: Define your messages list
        messages=[] 
    )
    
    return response.choices[0].message.content

# --- TEST YOUR FUNCTION ---
# email = generate_email(
#     subject="Project Update",
#     recipient_name="Sarah",
#     additional_info="We finished phase 1 ahead of schedule. Starting phase 2 next week.",
#     tone="enthusiastic"
# )
# print(email)
